# Feedlot V2 · Produksi 15.000

Targetnya 15.000 **placement baru** selama 2024–2025 dengan seed 42. Pilih **CPU / None** dan **High-RAM** pada **Runtime → Change runtime type**. Di laptop Windows, biarkan browser terbuka, gunakan AC, dan pastikan Sleep nonaktif. Layar boleh padam. Di iPhone, gunakan Safari dan pilih **Minta Situs Web Desktop** bila kontrol tersembunyi.

Satu START menjalankan operasi, penyimpanan checkpoint Drive, finance, penyimpanan checkpoint Drive, audit penuh, dan publikasi akhir. Jika Colab berhenti, checkpoint terakhir yang terverifikasi tetap bisa dipulihkan dari Drive. Fase yang belum tersegel dapat perlu diulang.

[Panduan produksi](https://github.com/wijaya-rama/portfolio-feedlot-management/blob/stage7-colab-live-monitoring/simulator_v2/stage6/colab/README_PRODUKSI.md) · [Runbook](https://github.com/wijaya-rama/portfolio-feedlot-management/blob/stage7-colab-live-monitoring/simulator_v2/stage6/colab/RUNBOOK_PRODUKSI.md)

In [ ]:
# @title 1 · Persiapan runtime
import os
from pathlib import Path
import subprocess
import sys

assert sys.platform == "linux" and Path("/content").is_dir(), "Jalankan di Colab Linux."
assert (3, 11) <= sys.version_info[:2] <= (3, 13), "Gunakan runtime Colab dengan Python 3.11–3.13."
REPOSITORY = "https://github.com/wijaya-rama/portfolio-feedlot-management.git"
SIMULATOR_SHA = "9afdbb6cf413395a3fb0b7b4ed5d2f0fa0e6da2e"
MONITOR_SHA = "22c6a2b47ca6433ddf13cb45a028d369142970a4"
ROOT = Path("/content/feedlot-v2-production")
SIMULATOR = ROOT / "source"
MONITOR = ROOT / "monitor"
VENV = ROOT / "venv"
WORK = ROOT / "runs"
PYTHON = VENV / "bin" / "python"
DRIVE_TEST_OK = PREFLIGHT_OK = POLICY_OK = PROGRESS_TEST_OK = False

def checkout(source, expected):
    source.mkdir(parents=True, exist_ok=True)

    def git(*args):
        return subprocess.check_output(["git", "-C", str(source), *args], text=True).strip()

    if not (source / ".git").is_dir():
        git("init", "--quiet")
    present = subprocess.run(["git", "-C", str(source), "cat-file", "-e", f"{expected}^{{commit}}"],
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    if present.returncode:
        git("fetch", "--no-tags", "--depth=1", REPOSITORY, expected)
        assert git("rev-parse", "FETCH_HEAD") == expected, "Commit hasil unduhan berbeda."
    git("checkout", "--detach", "--quiet", expected)
    assert git("rev-parse", "HEAD") == expected
    assert not git("status", "--porcelain", "--untracked-files=no"), "Checkout sumber telah berubah."

print("Menyiapkan simulator terpin dan kode monitoring.", flush=True)
checkout(SIMULATOR, SIMULATOR_SHA)
checkout(MONITOR, MONITOR_SHA)
sys.path.insert(0, str(MONITOR))
from simulator_v2.stage6.colab.notebook_stream import stream_command

def child_environment():
    return {**os.environ, "PYTHONTZPATH": "", "PYTHONHASHSEED": "42"}

requirements = SIMULATOR / "simulator_v2/stage6/colab/requirements-production.txt"
for command in (
    [sys.executable, "-m", "venv", str(VENV)],
    [str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check", "--no-cache-dir",
     "--only-binary=:all:", "-r", str(requirements)],
    [str(PYTHON), "-m", "pip", "check"],
    [str(PYTHON), "-c", "import pyarrow; from zoneinfo import ZoneInfo; "
     "assert pyarrow.Codec.is_available('zstd'); ZoneInfo('America/Edmonton')"],
):
    if stream_command(command, env=child_environment()) != 0:
        raise RuntimeError("Persiapan runtime gagal. Periksa output di atas.")

def jalankan(action, *options):
    command = [str(PYTHON), "-u", "-m", "simulator_v2.stage6.colab.production",
               action, "--local", str(WORK), "--commit", MONITOR_SHA,
               "--simulator-source", str(SIMULATOR), *options]
    return stream_command(command, cwd=MONITOR, env=child_environment()) == 0

print(f"Simulator terpin siap · {SIMULATOR_SHA}", flush=True)
print(f"Monitoring siap · {MONITOR_SHA}", flush=True)

## Akses Drive dan pemeriksaan awal

Otorisasi akun yang akan menyimpan checkpoint. Notebook menggunakan Drive API agar checksum berasal dari server Drive. Gunakan akun yang sama saat memulihkan. Uji koneksi menulis beberapa byte ke folder uji yang dipindahkan ke Sampah setelah verifikasi. Preflight membaca kuota Drive serta memeriksa RAM efektif, disk lokal dan jadwal 15.000 tanpa menjalankan simulasi.

In [ ]:
# @title 2 · Hubungkan Drive
from google.colab import auth

DRIVE_TEST_OK = PREFLIGHT_OK = False
auth.authenticate_user()
print("Otorisasi Colab selesai. Jalankan Uji koneksi Drive.")

In [ ]:
# @title 3 · Uji koneksi Drive
DRIVE_TEST_OK = jalankan("probe-drive")

In [ ]:
# @title 3b · Uji tampilan progres
PROGRESS_TEST_OK = False
demo = (
    "import sys, time\n"
    "print('UJI · URL folder akan ditampilkan di sini', flush=True)\n"
    "time.sleep(1)\n"
    "print('UJI · Fase operations dimulai', flush=True)\n"
    "time.sleep(1)\n"
    "print('UJI · Heartbeat · operations · 2 detik · checkpoint Drive belum ada', file=sys.stderr, flush=True)\n"
    "time.sleep(2)\n"
)
PROGRESS_TEST_OK = stream_command([str(PYTHON), "-u", "-c", demo], cwd=MONITOR, env=child_environment()) == 0
print("Pastikan heartbeat uji terlihat sebelum sel selesai. Uji ini tidak menjalankan simulator.", flush=True)

In [ ]:
# @title 4 · Preflight
PREFLIGHT_OK = jalankan("preflight")
if PREFLIGHT_OK:
    print("Preflight lulus. Tinjau sumber daya dan keputusan rilis sebelum START.")

## Keputusan rilis

Dengan mencentang persetujuan, Anda memilih rilis **dataset sintetis dengan integritas operasional dan finansial** setelah audit lengkap dan verifikasi Drive. Stage 5 tetap **belum dipromosikan**. Keputusan ini berlaku untuk percobaan produksi yang Anda jalankan atau lanjutkan.

In [ ]:
# @title 5 · Persetujuan rilis integritas
Setujui_rilis_integritas = False  # @param {type:"boolean"}
POLICY_OK = Setujui_rilis_integritas
print("Rilis integritas disetujui." if POLICY_OK else "Persetujuan belum diberikan.")

## Mulai produksi

Jalankan START hanya sekali untuk percobaan baru. Ketik `MULAI 15000` ketika siap. Salin ID folder Drive dari tautan yang tercetak. Setelah otorisasi itu, simpan dan verifikasi checkpoint serta publikasi akhir berjalan otomatis.

Sebelum START, jalankan **Uji tampilan progres** dan pastikan heartbeat uji muncul ketika sel masih aktif. Pada run sungguhan, heartbeat pertama biasanya muncul sekitar 15 detik setelah worker aktif, lalu sekitar tiap 120 detik.

In [ ]:
# @title 6 · START
Siap_mulai = False  # @param {type:"boolean"}

if Siap_mulai:
    if not (globals().get("DRIVE_TEST_OK") and globals().get("PREFLIGHT_OK") and globals().get("POLICY_OK")
            and globals().get("PROGRESS_TEST_OK")):
        raise RuntimeError("Selesaikan Uji koneksi Drive, Uji tampilan progres, Preflight, dan persetujuan rilis integritas.")
    if input("Ketik MULAI 15000 lalu tekan Enter. ").strip() == "MULAI 15000":
        jalankan("start", "--accept-integrity-only", "--confirm", "MULAI 15000")
    else:
        print("START dibatalkan.")
else:
    print("START menunggu otorisasi eksplisit.")

## Pantau, hentikan, atau lanjutkan

Buka [Monitor Drive](https://colab.research.google.com/github/wijaya-rama/portfolio-feedlot-management/blob/stage7-colab-live-monitoring/simulator_v2/stage6/colab/Feedlot_V2_Status.ipynb) di tab dan runtime terpisah. Masukkan ID folder untuk memeriksa status dan fase, heartbeat terakhir, klaim dan checkpoint. Anda juga bisa membuka `status.json` langsung di folder percobaan. Pastikan satu heartbeat nyata terlihat di sel produksi dan di Monitor Drive sebelum meninggalkan laptop.

Untuk menghentikan sel, tekan **Stop/Interrupt**. Tunggu sampai worker berhenti, lalu periksa klaim melalui Monitor Drive. Fase yang belum tersegel dapat perlu diulang.

In [ ]:
# @title 7 · Status
ID_folder_Drive = ""  # @param {type:"string"}

if ID_folder_Drive.strip():
    jalankan("status", "--folder", ID_folder_Drive.strip())
else:
    print("Masukkan ID dari tautan folder percobaan Drive.")

In [ ]:
# @title 8 · RESUME
ID_folder_Drive_resume = ""  # @param {type:"string"}

if ID_folder_Drive_resume.strip():
    if not globals().get("PROGRESS_TEST_OK"):
        raise RuntimeError("Jalankan Uji tampilan progres lebih dulu.")
    if not globals().get("POLICY_OK"):
        raise RuntimeError("Jalankan Persetujuan rilis integritas.")
    if input("Ketik LANJUT 15000 lalu tekan Enter. ").strip() == "LANJUT 15000":
        jalankan("resume", "--folder", ID_folder_Drive_resume.strip(),
                 "--accept-integrity-only", "--confirm", "LANJUT 15000")
    else:
        print("RESUME dibatalkan.")
else:
    print("Masukkan ID folder percobaan untuk melanjutkan.")

In [ ]:
# @title 9 · STOP/RECOVER
ID_folder_Drive_recover = ""  # @param {type:"string"}
Runtime_lama_sudah_berhenti = False  # @param {type:"boolean"}

if ID_folder_Drive_recover.strip() and Runtime_lama_sudah_berhenti:
    if not globals().get("PROGRESS_TEST_OK"):
        raise RuntimeError("Jalankan Uji tampilan progres lebih dulu.")
    if not globals().get("POLICY_OK"):
        raise RuntimeError("Jalankan Persetujuan rilis integritas.")
    if input("Ketik PULIHKAN 15000 lalu tekan Enter. ").strip() == "PULIHKAN 15000":
        jalankan("recover", "--folder", ID_folder_Drive_recover.strip(),
                 "--accept-integrity-only", "--confirm", "PULIHKAN 15000")
    else:
        print("RECOVER dibatalkan.")
else:
    print("Gunakan setelah RESUME melaporkan lock dan runtime lama sudah berhenti.")

## Audit dan publikasi akhir

Audit independen memeriksa operasi, finance dan 48 close. Paket final diterbitkan otomatis sesudah verifikasi inventaris Parquet, SHA-256 partisi, dan SHA-256 arsip di server Drive. Gunakan **Status** untuk memastikan `completed`, `validation_status=passed` dan SHA-256 `dataset-integrity-only.tar`. Laporan dan `dataset/release_manifest.json` tersedia di dalam arsip.

Colab menentukan umur runtime. Jendela sekitar 12 jam dapat berakhir sebelum paket final terverifikasi. Google mendokumentasikan maksimum 24 jam untuk Pro+ dengan unit komputasi yang cukup. Periksa paket yang aktif pada akun Anda sebelum mengandalkan eksekusi latar belakang. [FAQ Colab](https://research.google.com/colaboratory/faq.html?hl=en)